# 02 · Teste local (piloto, MPS)

Roda poucas imagens do **piloto** (excluídas das análises) em cada modelo local para conferir: o modelo carrega no MPS,
a resposta (grau ICDR + edema macular) é lida corretamente, o escore P(referenciável) não sai `nan`
e quanto tempo leva cada imagem.
Os modelos são carregados um de cada vez e liberados antes do próximo.

Antes: `huggingface-cli login` e aceite as licenças do MedGemma e do Gemma 3 no Hugging Face.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import config
pd.set_option("display.max_columns", 50, "display.width", 250, "display.max_rows", 200)
from vlmdr import cohort, infer, prompts

In [ ]:
MODELOS = [k for k, v in prompts.load_models().items() if v["local"]]   # ou uma lista, ex.: ["gemma3_4b"]
N = 20                      # imagens do piloto por base
PROMPT = "main_en"
print("modelos:", MODELOS, "| dispositivo:", infer.device_name())

In [ ]:
p = prompts.load_prompts()[PROMPT]
cohorts = {ds: cohort.load(ds) for ds in ("brset", "mbrset")}
out, info = [], []
for m in MODELOS:
    try:
        vlm = infer.VLM(m)
    except Exception as e:
        print(f"{m}: falhou ao carregar -> {type(e).__name__}: {e}")
        continue
    y_ids, n_ids = vlm.answer_ids(p)
    print(f"\n{m}: {vlm.cfg['hf_id']} | revisão {vlm.revision} | dtype {vlm.dtype_name}")
    print("  tokens dos graus:", [[vlm.tok.decode([i]) for i in ids] for ids in vlm.grade_ids()])
    print("  tokens edema sim:", [vlm.tok.decode([i]) for i in y_ids], "| não:", [vlm.tok.decode([i]) for i in n_ids])
    for ds, c in cohorts.items():
        pil = c[c.split == "pilot"].head(N)
        r = infer.run(vlm, pil, PROMPT, run_tag="piloto")
        out.append(r.merge(c[["image_id", "referable", "dr_icdr", "macular_edema"]], on="image_id"))
    info.append({"modelo": m, "memoria_GB": round(infer.peak_memory_gb(vlm.device), 1)})
    vlm.close()
out = pd.concat(out, ignore_index=True)
info = pd.DataFrame(info)

## Respostas por imagem

In [ ]:
out[["model", "dataset", "referable", "dr_icdr", "macular_edema", "raw_text", "grade_pred", "edema_pred",
     "answer", "p_yes", "grade_mass", "edema_mass", "seconds"]]

## Grau previsto × grau real (linhas = real, colunas = previsto)

In [ ]:
for m, g in out.groupby("model"):
    print(m)
    display(pd.crosstab(g.dr_icdr, g.grade_pred.fillna(-1).astype(int).rename("previsto (-1 = ilegível)")))

## Resumo por modelo (piloto: só confere o funcionamento, não estima desempenho)

In [ ]:
n_total = sum(len(c.query("split == 'eval'")) for c in cohorts.values())
out["yhat"] = (out.answer != "no").astype(int)       # inválida = encaminhar
rows = []
for m, g in out.groupby("model"):
    pos, neg = g[g.referable == 1], g[g.referable == 0]
    rows.append({"modelo": m, "n": len(g), "invalidas": int((g.answer == "invalid").sum()),
                 "p_yes_nan": int(g.p_yes.isna().sum()), "massa_graus": round(g.grade_mass.mean(), 3),
                 "massa_edema": round(g.edema_mass.mean(), 3), "edema_ilegivel": int(g.edema_pred.isna().sum()),
                 "seg_mediana": round(g.seconds.median(), 2),
                 "referenciaveis_encaminhados": f"{pos.yhat.sum()}/{len(pos)}",
                 "nao_ref_sem_encaminhar": f"{(neg.yhat == 0).sum()}/{len(neg)}"})
res = pd.DataFrame(rows).merge(info, on="modelo")
res["horas_por_prompt"] = (res.seg_mediana * n_total / 3600).round(1)
res

**Se P(sim) sair `nan` ou as respostas vierem estranhas no MPS** para um modelo, troque o tipo numérico só dele:
`vlm = infer.VLM("<modelo>", dtype="float16")` (ou `"float32"`, que usa o dobro de memória), rode a célula de inferência para ele e `vlm.close()`.

Os arquivos `*_piloto.csv` ficam em `results/preds/` e **não** entram na análise. A inferência retoma de onde parou:
para rodar de novo com outro prompt, mova os `*_piloto.csv` antigos para outra pasta.